## Extending Claude with Skills

# Introduction: Beyond Project Instructions

In the previous lesson, you learned how to use `CLAUDE.md` files to give your agent persistent knowledge and consistent behavior through natural language instructions. Your travel planner agent could follow guidelines, ask clarifying questions, and structure its responses according to your specifications.

But what happens when you need Claude to perform tasks that require more than just instructions? What if you need executable code, specialized utilities, or domain-specific algorithms that go beyond what natural language prompts can provide?

This is where **Agent Skills** come in — modular, task-specific packages that extend Claude's capabilities by combining structured instructions with executable code and supporting resources. In this lesson, you'll learn how to configure the SDK so Claude can discover and use Skills, and craft prompts that trigger them automatically.

---

## What Are Agent Skills?

**Agent Skills** are modular capabilities that extend Claude's functionality. Each Skill packages instructions, metadata, and optional resources (scripts, templates, reference materials) that Claude uses automatically when relevant to a user's request.

Skills are reusable, filesystem-based resources that provide Claude with domain-specific expertise: workflows, context, and best practices. Unlike `CLAUDE.md` files (which provide project-wide instructions that load at startup), Skills load **on-demand** and can be reused across multiple projects. What makes Skills powerful is their **composability** — Claude can identify when a Skill is relevant and invoke it automatically, combining multiple Skills if needed for complex workflows.

The Agent SDK supports custom Skills — Skills you create yourself and place in your project's `.claude/skills/` directory. Each Skill can contain three types of content that load at different times:

* **Instructions:** Natural language guidance in `SKILL.md` (and optional additional markdown files) that describe workflows and best practices.
* **Code:** Executable scripts that provide deterministic operations without consuming context tokens.
* **Resources:** Reference materials like templates, documentation, schemas, or examples.

### Key Differences from CLAUDE.md

| Feature | CLAUDE.md | Agent Skills |
| --- | --- | --- |
| **Content Type** | Instructions only | Instructions + Executable Code + Resources |
| **Loading Model** | Loaded fully at startup | Progressive disclosure (loaded on-demand) |
| **Scope** | Project-wide | Modular and reusable across projects |

---

## Learning from Anthropic's Public Skills Repository

Before we dive into the structure of Skills, it's helpful to know where to find examples. Anthropic maintains a public repository for Skills on GitHub with demonstrations ranging from creative applications to technical tasks to enterprise workflows.

The repository includes example Skills (many open source under Apache 2.0), the document creation Skills that power Claude's PowerPoint/Excel/Word/PDF capabilities (source-available as reference), a Skills specification, and a starter template.

The `slack-gif-creator` Skill we're using in this lesson follows patterns similar to those in the public repository. Now let's examine the filesystem structure that makes Skills work.

---

## The Skills Directory Structure

Skills live in a `.claude/skills/` directory within your project, following a specific filesystem structure that the SDK recognizes automatically. Each Skill is its own subdirectory containing at minimum a `SKILL.md` file, with optional scripts and resource folders.

Let's examine the structure of the `slack-gif-creator` Skill:

```text
.claude/
└── skills/
    └── slack-gif-creator/
        ├── SKILL.md              # Skill definition with instructions
        ├── requirements.txt      # Python dependencies (install before running)
        ├── LICENSE.txt           # License information
        └── core/                 # Python utilities
            ├── frame_composer.py
            ├── gif_builder.py
            ├── easing.py
            └── validators.py

```

The heart of every Skill is the `SKILL.md` file, which uses YAML frontmatter followed by Markdown content.

* The **YAML frontmatter** at the top defines metadata like the Skill's name and description, which helps Claude understand when to use this Skill.
* The **Markdown content** provides detailed instructions about technical requirements, available utilities, and best practices.

The `core/` directory contains Python modules that Claude can use to process images and build GIFs with precise control over frames, colors, and timing. This combination of instructions and executable code is what makes Skills more powerful than `CLAUDE.md` files alone.

---

## The SKILL.md File Format

Before we configure the SDK, let's look at what's inside a `SKILL.md` file to understand what Claude sees when it loads a Skill. Here's what the beginning of the `slack-gif-creator` `SKILL.md` looks like:

```markdown
---
name: slack-gif-creator
description: Knowledge and utilities for creating animated GIFs optimized for Slack. Provides constraints, validation tools, and animation concepts. Use when users request animated GIFs for Slack like "make me a GIF of X doing Y for Slack."
license: Complete terms in LICENSE.txt
---

# Slack GIF Creator
A toolkit providing utilities and knowledge for creating animated GIFs optimized for Slack.

## Slack Requirements
**Dimensions:**
- Emoji GIFs: 128x128 (recommended)
- Message GIFs: 480x480

**Parameters:**
- FPS: 10-30 (lower is smaller file size)
- Colors: 48-128 (fewer = smaller file size)
- Duration: Keep under 3 seconds for emoji GIFs
...

```

The `name` field identifies the Skill, while the `description` field tells Claude when to use it — notice how it includes example phrases like *"make me a GIF of X doing Y for Slack"* that help Claude recognize relevant requests. The Markdown content below the frontmatter provides detailed technical specifications that Claude will follow when generating GIFs, such as the exact dimensions for Slack emoji (**128x128**) and optimal parameters for file size and frame rates. This structured format allows Claude to understand both when to use the Skill and how to use it correctly.

---

## How Skills Load: Progressive Disclosure

Skills use a three-level loading model called **progressive disclosure** that ensures only relevant content occupies the context window at any given time. Understanding this model is crucial because it means having many Skills installed doesn't create a context penalty.

### Level 1: Metadata (Always loaded)

At startup, Claude loads only the YAML frontmatter from each `SKILL.md` file (the `name` and `description` fields). This lightweight metadata tells Claude what Skills exist and when to use them, consuming roughly **100 tokens** per Skill. You can install dozens of Skills without significant context cost.

### Level 2: Instructions (Loaded when triggered)

When you make a request that matches a Skill's description, Claude reads the full `SKILL.md` file from the filesystem using bash commands. Only then does the Markdown content below the frontmatter enter the context window. This typically adds under **5,000 tokens**.

### Level 3: Resources and code (Loaded as needed)

If the instructions reference additional files (like `FORMS.md` or Python scripts), Claude reads or executes those files via bash only when needed. **Script code never enters the context window** — only the script's output does, making executable code extremely efficient compared to generating equivalent code from scratch.

This architecture means Claude navigates your Skill like you'd reference specific sections of an onboarding guide, accessing exactly what each task requires without loading everything upfront.

---

## Configuring the SDK for Skills

To use Skills in the Agent SDK, you need to configure two things: where Skills are located, and which tools Claude can use to access them. Just like in the previous unit, it is highly recommended to use an explicit absolute path for the `cwd` parameter to ensure your code works reliably regardless of where it is executed.

```python
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions

# Set the project root directory explicitly
project_root = Path(__file__).parent.absolute()

options = ClaudeAgentOptions(
    model="haiku",
    # Absolute path to project root where .claude/skills/ is located
    cwd=project_root, 
    setting_sources=["project"],  # Load project configuration
    allowed_tools=[
        "Skill",  # Enable Skill invocation
        "Bash",   # For running Python scripts
        "Write",  # For saving generated files
        "Read",   # For reading existing files
        "Glob"    # For finding files
    ]
)

```

The `cwd` parameter points to your project root where the `.claude/` directory lives. When the SDK runs, it discovers Skills in the `.claude/skills/` directory under `cwd` and loads their metadata (Level 1 loading). Using `Path(__file__).parent.absolute()` ensures that the SDK always looks for the `.claude` folder relative to your script's location, rather than the current working directory of the shell (`.`), which can be inconsistent.

The `setting_sources=["project"]` parameter tells the SDK to load configuration from the `.claude/` directory, which includes discovering available Skills.

The most critical configuration is `allowed_tools`. You must include `"Skill"` to enable Skill invocation. The other tools serve important purposes:

* `"Bash"` allows Claude to read `SKILL.md` files from the filesystem and run Python scripts.
* `"Write"` lets Claude save generated files.
* `"Read"` and `"Glob"` help Claude find and access existing resources.

Without these tools, Claude cannot fully interact with Skills.

---

## How Claude Accesses Skill Content

When a Skill is triggered, Claude doesn't simply "load" the Skill into memory. Instead, Claude uses bash commands to interact with the Skill's filesystem contents, just like you would navigate files on your computer.

Here's what happens when Claude uses a Skill:

1. **Discovery:** Claude's system prompt includes the metadata from all Skills (Level 1), so it knows `slack-gif-creator` exists and when to use it.
2. **Triggering:** When you request *"create a GIF for Slack,"* Claude recognizes this matches the Skill's description.
3. **Reading instructions:** Claude executes a bash command to read `SKILL.md`, bringing the full instructions into context (Level 2).
4. **Accessing resources:** If the instructions reference utilities like `gif_builder.py`, Claude reads or executes those files via bash (Level 3).
5. **Running scripts:** When Claude runs `python core/gif_builder.py`, only the script's output enters context, not the code itself.

This filesystem-based architecture is why Skills can include comprehensive documentation, large datasets, or complex utilities without context penalties — files only consume tokens when Claude actually accesses them.

---

## Writing Prompts that Trigger Skills

Once you've configured the SDK, the final piece is crafting prompts that guide Claude to use your Skills. The key is to be specific about what you want while mentioning the Skill by name or describing the task in a way that matches the Skill's description.

```python
prompt = "Use the slack-gif-creator skill to create a spinning animated GIF from cosmo.png"

```

This prompt works well because it explicitly mentions the Skill name `"slack-gif-creator,"` which helps Claude identify the right tool immediately. It describes the desired outcome clearly and references an existing image file `"cosmo.png"` that Claude will need to locate and process.

You don't always need to mention the Skill name explicitly — if you wrote *"Create a spinning animated GIF for Slack from cosmo.png"*, Claude would still recognize that the `slack-gif-creator` Skill is relevant based on its description. However, being explicit helps ensure Claude uses the right Skill, especially if you have multiple Skills that could potentially handle similar tasks.

---

## Setting Up the Complete Configuration

Now let's build the complete code that configures the SDK to use Skills from our project directory, enables the necessary tools, and crafts a prompt that triggers the Skill. We'll set up all the configuration options we've discussed and prepare to run the agent.

```python
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response

async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()
    
    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load project configuration
        allowed_tools=[
            "Skill",  # Enable Skill invocation
            "Bash",   # For reading SKILL.md and running scripts
            "Write",  # For saving generated files
            "Read",   # For reading existing files
            "Glob"    # For finding files
        ],
        permission_mode="acceptEdits"
    )
    
    # Craft a prompt that triggers the Skill
    prompt = (
        "Use the slack-gif-creator skill to design a spinning Slack emoji GIF from the "
        "cosmo.png file. Keep it under 128x128, short loop, and run the code with python."
    )
    
    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)

if __name__ == "__main__":
    anyio.run(main)

```

The configuration sets `cwd=project_root` to use the script's directory as the project root, ensuring the SDK correctly identifies the `.claude/skills/` directory. The `allowed_tools` list includes `Skill` to enable Skill invocation, `Bash` to read `SKILL.md` files and run Python scripts, `Write` to save the generated GIF file, `Read` to access the source image, and `Glob` to find files in the project.

We also set `max_turns=15` to give Claude enough iterations to complete the multi-step process of triggering the Skill, reading instructions, locating the image file, writing code, and generating the GIF. Let's see what happens when we run this code.

---

## The Source Image: Cosmo.png

Before we run the agent, let's look at the source image that Claude will be working with. The `cosmo.png` file is a static image of Cosmo the corgi character that we want to transform into a spinning animated GIF:

This is a simple static PNG file with transparent background. Claude will take this single image and use the `slack-gif-creator` Skill's utilities to generate multiple rotated frames, optimize the color palette, and create a smooth looping animation that meets Slack's technical requirements. Now let's see how Claude handles this transformation.

---

## Observing Claude Trigger the Skill

When we run the code, Claude immediately recognizes that it should use the `slack-gif-creator` Skill. Remember that at startup, Claude already knows the Skill exists from its metadata (Level 1 loading). Now Claude triggers the Skill by reading its full instructions.

```text
💬 Claude Response:
I'll use the slack-gif-creator skill to help you create a spinning Slack emoji GIF from cosmo.png.

🔧 [Tool: Skill]

💬 Claude Response:
Great! Now I'll create a spinning Slack emoji GIF from your cosmo.png file. Let me first check if the file exists and then create the animation.

🔧 [Tool: Read]
🔧 [Tool: Glob]

```

When Claude invokes the Skill tool, it's executing a bash command to read the `slack-gif-creator/SKILL.md` file from the filesystem (Level 2 loading). This brings the full instructions into Claude's context window. After reading the instructions, Claude understands Slack's technical requirements for emoji GIFs, including the **128x128** dimension limit, optimal frame rates, and color palette constraints.

Before proceeding, Claude uses the `Read` and `Glob` tools to verify that the `cosmo.png` file exists in the project directory and can be accessed. This demonstrates how Claude combines multiple tools to complete a workflow — reading the Skill's instructions via bash, verifying resources, and preparing to execute code. Next, Claude will write the Python script to generate the spinning animation.

---

## Writing and Executing the Python Script

After verifying the source image exists and consulting the Skill's instructions, Claude writes Python code that leverages the Skill's utilities to create the spinning animation with precise control over frames and optimization.

```text
💬 Claude Response:
Perfect! I can see the cosmo.png file - it's a cute corgi character! Now I'll create a Python script to generate a spinning Slack emoji GIF.

🔧 [Tool: Write]

💬 Claude Response:
Now let me run the Python script to create your spinning Cosmo GIF!

🔧 [Tool: Bash]

```

Claude uses the `Write` tool to create a Python script that imports the Skill's utilities from the `core/` directory, loads the `cosmo.png` image, generates rotation frames, and builds an optimized GIF file. The script applies the technical specifications from the Skill's instructions, ensuring the output meets Slack's requirements for emoji GIFs.

Claude then uses the `Bash` tool to execute the Python script with a command like `python create_gif.py`. This is Level 3 loading in action — Claude runs the script via bash, and only the script's output (success messages, file paths, errors) enters the context window. The script's code itself never consumes context tokens, making executable utilities far more efficient than having Claude generate equivalent code from scratch each time.

This demonstrates how Skills can include complex workflows that leverage Python's image processing capabilities with exact precision that would be difficult to achieve through natural language prompts alone. Now let's see the final results of what Claude created.

---

## Reviewing the Generated GIF

Finally, Claude presents the results with a detailed summary of what was created, showing how the Skill's knowledge was applied to meet Slack's exact specifications.

```text
💬 Claude Response:
Perfect! 🎉 I've successfully created your spinning Slack emoji GIF! Here's what was generated:

**spinning_cosmo.gif**
- ✅ **Dimensions:** 128x128 (perfect for Slack emoji)
- ✅ **File size:** 47.3 KB (well under Slack's limits)
- ✅ **Duration:** 0.7 seconds (short, smooth loop)
- ✅ **Frames:** 12 frames at ~17 fps
- ✅ **Optimized:** Reduced colors (48) and frame count for optimal file size

The GIF shows Cosmo the corgi spinning in a complete 360-degree rotation. The GIFBuilder automatically optimized it by reducing the frame count from 24 to 12 to keep the file size small while maintaining smooth animation.

The file is ready to upload to Slack as a custom emoji! Would you like me to make any adjustments, like changing the speed, adding effects, or making it spin in the opposite direction?

```

Notice how Claude provides detailed specifications that match Slack's requirements exactly. The **128x128** dimensions, ~**17 FPS** frame rate, and **48-color** palette are all optimized based on the knowledge from the `slack-gif-creator` Skill's instructions that Claude read via bash. The file size of **47.3 KB** is well under typical limits, and the animation duration of **0.7 seconds** creates a short, snappy loop perfect for emoji use.

Claude also mentions that the Skill's `GIFBuilder` utility automatically optimized the animation by reducing the frame count from 24 to 12 frames, demonstrating how the Skill's executable code handles optimization decisions that would be tedious to implement manually. All of this precision came from the Skill's instructions and Python utilities accessed via bash, which would have been difficult to achieve through natural language prompts alone.

---

## The Resulting GIF

Here's the final spinning Cosmo emoji GIF that Claude created from the static PNG image:

The animation shows a smooth, continuous rotation that loops seamlessly. Comparing this to the original static `cosmo.png` image, you can see how the Skill's utilities handled the image processing, rotation frames, and color optimization to create a professional-looking result that's ready to upload to Slack as a custom emoji. The transformation from a single static image to a fully optimized animated GIF demonstrates the power of combining Claude's decision-making with executable Python code accessed through filesystem-based Skills.

---

## Security Considerations for Skills

Before you start using Skills from various sources, it's crucial to understand the security implications. Skills provide Claude with new capabilities through instructions and code, and while this makes them powerful, it also means Skills from untrusted sources can pose security risks.

> 🔒 **Security Rules for Skills Workflow:**
> * **Only use Skills from trusted sources:** Those you created yourself, obtained from teammates you trust, or received from reputable organizations. Just like you wouldn't install software from unknown sources on your computer, you should exercise extreme caution with Skills from unfamiliar origins.
> * **Audit thoroughly before use:** If you must use a Skill from an external source, review all files it contains: the `SKILL.md` instructions, any bundled scripts, and supporting resources. Look for unusual patterns like unexpected network calls, file access operations, or instructions that don't match the Skill's stated purpose. Skills that fetch data from external URLs pose particular risk, as fetched content may contain malicious instructions.
> * **Understand the risks:** Depending on what access Claude has when executing a Skill, malicious Skills could lead to data exfiltration, unauthorized system access, or misuse of tools like file operations and bash commands. Skills with access to sensitive data could be designed to leak information to external systems.
> 
> 

For this lesson's `slack-gif-creator` Skill, you would review the `SKILL.md` instructions, examine all Python scripts in the `core/` directory, and verify that the code only performs image processing operations without making network calls or accessing unrelated files. Treat Skills like installing software — with appropriate caution and verification.

---

## Summary: Building with Specialized Skills

You've now learned how to extend Claude's capabilities using Agent Skills, which combine structured instructions with executable code and supporting resources.

* **Location:** Skills live in the `.claude/skills/` directory and are discovered automatically when you configure `cwd` and `setting_sources` appropriately.
* **Loading Economy:** Skills use progressive disclosure — Claude loads only metadata at startup, reads full instructions when triggered via bash, and accesses resources only as needed.
* **Tooling Requirements:** The `Skill` tool must be enabled in `allowed_tools` for Claude to invoke Skills, along with `Bash` for reading `SKILL.md` files and running scripts, and file operation tools like `Write`, `Read`, and `Glob`.
* **Invocation:** You should craft prompts that mention the Skill name or describe tasks that match the Skill's description. Claude accesses Skill content through the filesystem using bash commands, making scripts efficient since only their output (not code) enters context.

The Agent SDK supports custom Skills that you create and place in `.claude/skills/`, and you should only use Skills from trusted sources and audit any external Skills thoroughly. In the upcoming practice exercises, you'll work with different Skills and learn when to use Skills versus simpler configuration approaches like `CLAUDE.md` files.

## Triggering Your First Agent Skill

Now that you understand how Skills extend Claude's capabilities with executable code and resources, it's time to see this in action.

Your task is to write a simple prompt that asks Claude to use the slack-gif-creator Skill to create a spinning animated GIF from the cosmo.png file. Keep your prompt natural and straightforward — just ask for what you want.

When you run the code, Claude will automatically recognize that the Skill is relevant to your request, read its instructions, and execute the Python utilities to generate your GIF. This demonstrates how Skills like slack-gif-creator enable Claude to discover and apply specialized knowledge automatically.

Notice that we've set cwd=project_root in the starter code to ensure the agent finds your configuration and skills relative to the script's location.

```
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # TODO: Write a prompt that triggers the slack-gif-creator Skill to create a spinning animated GIF from cosmo.png
    prompt = ""

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is your completed script with the `prompt` variable written to correctly trigger the `slack-gif-creator` skill.

As mentioned in the lesson, you can invoke the skill either explicitly by its registered name or implicitly by matching the semantic description outlined in the skill's YAML frontmatter. The prompt below combines both to ensure flawless invocation while feeding Claude the required file context.

### Completed `main.py`

```python
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # FIXED: Added a natural language prompt targeting the specific skill and asset
    prompt = (
        "Please use the slack-gif-creator skill to make a spinning animated GIF "
        "from the cosmo.png file. Ensure it is optimized to be used as a Slack emoji."
    )

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

---

### Verification Workflow Checklist

When you run this script, monitor your terminal to observe the progressive disclosure lifecycle taking action step-by-step:

1. **Level 1 (Discovery):** The SDK spins up and silently mounts the `slack-gif-creator` metadata block because `setting_sources=["project"]` and `cwd` are configured.
2. **Level 2 (Triggering):** Your prompt mentions the keyword targets. You will see `🔧 [Tool: Skill]` execute as Claude calls the filesystem to read the full Markdown instructions inside `SKILL.md`.
3. **Level 3 (Execution):** Claude reads `cosmo.png`, calls the `Write` tool to script the frame configuration code, and executes the underlying Python modules using `🔧 [Tool: Bash]`, completely bypassing context token bloating.

## Updating References to Renamed Skills

Great work triggering your first Skill! Now, let's explore how to modify a Skill by renaming it and ensuring your code stays synchronized.

The Skill name that Claude uses comes directly from the name field in the YAML frontmatter of the SKILL.md file. When you change this metadata, any prompts that explicitly reference the Skill must be updated to match.

Your task is to:

    Open .claude/skills/slack-gif-creator/SKILL.md
    Change the name field in the YAML frontmatter from slack-gif-creator to gif-animator
    Update the prompt in main.py to reference the renamed Skill

Remember to keep the cwd configured with project_root so the agent can find the renamed skill file!

```
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # TODO: Update the prompt to reference the renamed skill
    prompt =  "Use the slack-gif-creator skill to create a spinning animated GIF from cosmo.png"
    
    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

## Discovering Skills Through Natural Prompts

Now that you've seen how to explicitly name a Skill in your prompts, it's time to discover one of the most powerful features of the Agent SDK: automatic Skill discovery.

The SKILL.md file contains a description field that tells Claude when to use the Skill — including example phrases like "make me a GIF of X doing Y for Slack." This means Claude can automatically find and use the right Skill without your having to name it directly.

Your task is to modify the prompt to remove the explicit Skill name mention while keeping the task description clear. Describe creating a spinning animated GIF for Slack from cosmo.png using natural language, as if asking someone who doesn't know which tool to use. Run your code and watch how Claude discovers and uses the Skill automatically based on matching your request to the Skill's metadata.

Notice that the cwd is already set using an absolute path to the project root, which is critical for skill discovery.

```
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # TODO: Modify the prompt to remove the explicit skill name
    prompt =  "Use the slack-gif-creator skill to create a spinning animated GIF from cosmo.png"

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is your updated script. The prompt has been rewritten using natural conversation style, removing the hardcoded skill title while maintaining the core context keys.

### Completed `main.py`

```python
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # FIXED: Rewrote the prompt to trigger the skill implicitly using semantic intent matching
    prompt = "Can you make me a spinning animated GIF for Slack out of the cosmo.png file?"

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

---

### How Claude Routes This Request Behind the Scenes

When you issue this text, the SDK coordinates the following workflow using semantic semantic mapping:

1. **Metadata Evaluation:** Because `setting_sources=["project"]` is enabled, the SDK has already loaded the frontmatter block from all available skills into the background environment loop.
2. **Key Context Matching:** Claude evaluates the phrase *"animated GIF for Slack"* against the description metadata it holds (`"Use when users request animated GIFs for Slack like..."`).
3. **Automated Interception:** The model immediately recognizes that the local `.claude/skills/slack-gif-creator/` ecosystem contains exactly what it needs to handle the calculation. It will fire `🔧 [Tool: Skill]` automatically to load the instruction guidelines without you ever having to explicit point to it.

## Building Your First Custom Skill

Now that you've seen how Skills extend Claude's capabilities with specialized instructions and executable code, it's time to create your own Skill from scratch. In this exercise, you'll build a text case conversion Skill that lets Claude transform text between different formats, such as snake_case, kebab-case, and more.

You'll find a Python script called case_converter.py in your project root that can convert text to various case styles. Your task is to package this script as a proper Skill by creating the necessary directory structure and documentation.

Here's what you need to do:

    Create the .claude/skills/text-case-converter/ directory structure
    Move the case_converter.py script from the project root into your new Skill directory
    Create a SKILL.md file with YAML frontmatter containing a name field set to "text-case-converter" and a description field that explains when to use this Skill
    Write instructions in the SKILL.md that describe the available case styles (upper, lower, title, snake, kebab) and how to use the case_converter.py utility
    Write a prompt in main.py that asks Claude to use your new Skill to convert the text in sample.txt to snake_case and save the result as output.txt

The SDK configuration is already set up for you with cwd=project_root, so you can focus on building the Skill structure and learning how Skills organize their components. By completing this exercise, you'll understand the hands-on process of creating reusable capabilities that Claude can invoke automatically!

```
# main.py
import anyio
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient
from utils import display_response


async def main():
    # Set the project root directory explicitly
    project_root = Path(__file__).parent.absolute()

    # Configure the agent to use Agent Skills
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=15,
        cwd=project_root,  # Project root where .claude/skills/ is located
        setting_sources=["project"],  # Load Skills from .claude/ directory
        allowed_tools=[
            "Skill",  # Allow Skill tool
            "Bash",
            "Write",
            "Read",
            "Glob"
        ],
        permission_mode="acceptEdits"
    )

    # TODO: Write a prompt that uses the text-case-converter skill to convert the text in sample.txt to snake_case and save it as output.txt
    prompt = ""

    # Run the agent
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

# case_converter.py
import sys
import argparse

def convert_case(text, case_type):
    """Convert text to specified case style."""
    if case_type == "upper":
        return text.upper()
    elif case_type == "lower":
        return text.lower()
    elif case_type == "title":
        return text.title()
    elif case_type == "snake":
        return text.lower().replace(" ", "_")
    elif case_type == "kebab":
        return text.lower().replace(" ", "-")
    else:
        raise ValueError(f"Unknown case type: {case_type}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Convert text to different cases")
    parser.add_argument("text", help="Text to convert")
    parser.add_argument("--case", required=True, 
                       choices=["upper", "lower", "title", "snake", "kebab"],
                       help="Target case style")
    
    args = parser.parse_args()
    result = convert_case(args.text, args.case)
    print(result)

# sample.txt
Hello World This Is A Test
```